In [0]:
import pandas as pd
import numpy as np

# loading the data
extraction_df_spark=spark.read.table("mthimunye.default.gold_mineral_processing")

# converting pyspark into pandas dataframe
extraction_df=extraction_df_spark.toPandas()

display(extraction_df.head(5))


In [0]:
extraction_df.shape
#the output reveal 36 columns and 5 rows

In [0]:
extraction_df.info()

#the output reveal a null in pH column 

In [0]:
extraction_df.duplicated().sum()

In [0]:
# Show all rows that are duplicates,
duplicates = extraction_df[extraction_df.duplicated(keep=False)]
display(duplicates)


In [0]:
extraction_df.describe()

In [0]:
# Fill missing pH with the median
pH_median = extraction_df["pH"].median()
extraction_df["pH"] = extraction_df["pH"].fillna(pH_median)

In [0]:
# checking if the null is successfully replaced
extraction_df.info()

In [0]:
#Remove duplicate rows and update the DataFrame
extraction_df = extraction_df.drop_duplicates()

In [0]:
extraction_df.duplicated().sum()

**What is categorical encoding?**

Categorical encoding is the process of converting categorical data (text labels) into numerical values that machine-learning algorithms can process.

The variable we need to encode is ore_type, because it contains text categories.

We have converted a nominal categorical variable into numerical features using **one-hot encoding**. This makes the ore-type information usable by our machine-learning model without imposing an artificial ranking on the categories.

In [0]:
# Display the unique ore types
print(extraction_df["ore_type"].unique())

# Count the number of observations in each category
print(extraction_df["ore_type"].value_counts())

In [0]:
extraction_encoded = pd.get_dummies(
    extraction_df,
    columns=["ore_type"],
    drop_first=False,
    dtype=int
)

display(extraction_encoded.head())

**What is feature scaling?**

Feature scaling is the process of adjusting numerical variables so that they have comparable scales or ranges.

Notice that the values have different numerical ranges.
- Feed grade is relatively small.
- pH is around 10.
- Cyanide dosage is in the hundreds.

Some machine-learning algorithms are sensitive to these differences.

Without scaling, cyanide dosage can dominate the distance calculation simply because its numerical values are much larger.

This can cause the model to give too much influence to cyanide dosage, even when the other features are also important.

Important: We will not scale gold_recovery_pct because it is our target variable. We only scale the input features.

In [0]:
#separate input features (X) from the target (y)

X = extraction_encoded.drop(columns=["gold_recovery_pct"])

y = extraction_encoded["gold_recovery_pct"]


- X contains the input variables used to predict recovery
- y contains the gold recovery values we want to predict

In [0]:
# Split into training and testing data

from sklearn.model_selection import train_test_split
# X = input features; y = actual gold recovery (%)
# Keep 20% of observations unseen for testing.

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)


In [0]:
# Define the numerical input features that need scaling.
numerical_features = [
    "feed_grade_gpt",
    "pH",
    "cyanide_dose_mgL"
]

In [0]:
from sklearn.preprocessing import StandardScaler

# Step 1: Select the numerical features to scale.
numerical_features = [
    "feed_grade_gpt",
    "pH",
    "cyanide_dose_mgL"
]

# Step 2: Create the StandardScaler object.
scaler = StandardScaler()

# Step 3: Make copies of the training and testing datasets.
X_train = X_train.copy()
X_test = X_test.copy()

# Step 4: Learn the mean and standard deviation from training data.
# Then use them to scale the training features.
X_train[numerical_features] = scaler.fit_transform(
    X_train[numerical_features]
)

# Step 5: Scale testing data using the SAME training parameters.
# We use transform(), not fit_transform(), to prevent data leakage.
X_test[numerical_features] = scaler.transform(
    X_test[numerical_features]
)

# Step 6: Display the scaled training data.
display(X_train.head())

Notice the difference:
- fit_transform() learns the scaling parameters from training data and applies them.
- transform() applies those same parameters to test data without learning anything new.

When should you use each?
- **StandardScaler**: Commonly used for KNN, logistic regression and SVM.
- **MinMaxScaler**: Useful when you want features in a fixed range, particularly for some neural networks and distance-based methods.

**Robust Scaling using RobustScaler**

The final scaling method is RobustScaler.
1. What is RobustScaler?
RobustScaler is a scikit-learn tool that scales numerical features using:
- Median
- Interquartile range (IQR)
Unlike StandardScaler, it does not use the mean and standard deviation.

This makes it less sensitive to outliers.
2. What is an outlier?
An outlier is a numerical observation that is unusually far from the other observations in a dataset.

For example, consider cyanide dosage:

200, 210, 220, 230, 240, 1500

The value 1500 is much higher than the other observations. It could be an outlier.

In a real processing plant, an unusual cyanide dosage could represent a genuine operating event, a measurement error or a data-entry mistake. We should investigate before removing it.

![image_1793808718341.png](./image_1793808718341.png "image_1793808718341.png")

**Why do we split a dataset?**
- Training set: Data used by the model to learn patterns and relationships.
- Testing set: Data held back to evaluate how well the trained model performs on unseen observations.

A common approach is:
- 80% for training.
- 20% for testing.

**What is random_state?**
random_state controls the random selection so that you can reproduce the same split.

**What is stratify?**
Stratification ensures that the proportions of different classes are maintained in the training and testing sets.



In [0]:
train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

Our target, gold_recovery_pct, is **continuous**. We are predicting a numerical percentage, so this is a regression problem. You would not normally use stratify=y directly with a continuous target.

Stratification is primarily used for classification. For our current regression dataset, leave it out.

**What is data leakage?**
Data leakage happens when information that should not be available during model training influences the training process.

Consider our scaling exercise.

If you scale the entire dataset before splitting it, the scaler learns the mean and standard deviation using both training and testing observations.

In [0]:
from sklearn.linear_model import LinearRegression
# Create a linear regression model.(my algorithm)
model = LinearRegression()

# Learn how the input features relate to gold recovery.
model.fit(X_train, y_train)
print("Model training completed.")

In [0]:
# predict() estimates the recovery percentage for every test observation.

import pandas as pd
# Predict numerical gold recovery percentages.
y_pred = model.predict(X_test)

# Compare predictions with the actual test measurements.
results = pd.DataFrame({
    "Actual (%)": y_test.to_numpy(),
    "Predicted (%)": y_pred
})
display(results.round(2))

In [0]:
import pandas as pd

# Step 1: Enter new operating conditions.
new_sample = pd.DataFrame({
    "ore_type": ["Oxide"],
    "feed_grade_gpt": [0.35],
    "pH": [10.5],
    "cyanide_dose_mgL": [120]
})

# Step 2: Create input columns matching the training data.
new_input = pd.DataFrame(
    0.0, index=[0], columns=X_train.columns
)

# Copy the numerical feature values.
for col in numerical_features:
    new_input[col] = new_sample[col].values

# Encode the ore type using the training column names.
ore = new_sample["ore_type"].iloc[0]

if ore not in extraction_df["ore_type"].unique():
    raise ValueError("Unknown ore type")

ore_column = "ore_type_" + ore

if ore_column in new_input.columns:
    new_input[ore_column] = 1

# Step 3: Scale numerical features using the trained scaler.
new_input[numerical_features] = scaler.transform(
    new_input[numerical_features]
)

# Step 4: Predict gold recovery using the trained model.
prediction = model.predict(new_input)

# Step 5: Display the operating conditions and predicted recovery.
print("New operating conditions:")
display(new_sample)

print(f"Predicted Gold Recovery: {prediction[0]:.2f}%")

**Evaluate Your Linear Regression Model**

Evaluate how close the predicted gold recovery percentages are to the actual values.

In [0]:
import numpy as np
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score
)
# MAE: average absolute prediction error.
mae = mean_absolute_error(y_test, y_pred)

# RMSE: penalises large prediction errors more strongly.
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

# R-squared: compares the model with a mean-value baseline.
r2 = r2_score(y_test, y_pred)
print(f"MAE: {mae:.2f} percentage points")
print(f"RMSE: {rmse:.2f} percentage points")
print(f"R-squared: {r2:.3f}")

Imagine the actual gold recovery at a plant is 90%.
- MAE = 1.67: On average, your model makes an error of 1.67 percentage points. For example, it might predict 88.33% instead of 90%.
- RMSE = 1.96: Your model's errors are generally small, but this measurement gives more importance to bigger mistakes.
- R² = 0.874: Your model performs much better than simply predicting the average gold recovery every time. It reduces squared prediction errors by 87.4% compared with that baseline.`

In [0]:
import matplotlib.pyplot as plt
# Dots closer to the dashed line mean better predictions.

plt.scatter(y_test, y_pred)
low = min(y_test.min(), y_pred.min())
high = max(y_test.max(), y_pred.max())
plt.plot([low, high], [low, high], "k--")
plt.xlabel("Actual gold recovery (%)")
plt.ylabel("Predicted gold recovery (%)")
plt.title("Actual vs predicted recovery")
plt.grid(alpha=0.3)
plt.show()

In [0]:
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
from sklearn.inspection import permutation_importance

# Calculate feature importance
importance = permutation_importance(
    model, X_test, y_test,
    n_repeats=30,
    random_state=42,
    scoring="neg_mean_absolute_error"
)

# Plot feature importance
plt.figure(figsize=(8, 5))
plt.bar(X_test.columns, importance.importances_mean)

plt.xlabel("Features")
plt.ylabel("Importance Score (Increase in MAE)")
plt.title("Linear Regression Feature Importance")

plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

- Ore Type (Oxide): The most important feature, with the highest importance score (1.68).
- Cyanide Dosage: Has a significant influence on the model's predictions (0.70).
- Ore Type (Sulphide): Also contributes noticeably to predicting gold recovery (0.69).
- Feed Grade: Has a moderate importance score (0.54).
- pH: Has very little importance, suggesting the model does not rely strongly on it.
- Ore Type (Transition): Has a negative importance score, meaning shuffling this feature slightly improved predictions.
Overall: The model relies most on oxide ore type, followed by cyanide dosage and sulphide ore type, when predicting gold recovery.

Note: These scores show predictive importance, not necessarily cause and effect. The rankings may also be unstable because the test dataset is small